# Part 1 — Data Audit, EDA & Business Understanding
This notebook is generated by `build_part1.py`. It now documents package inspection, data quality checks, exploratory analysis, churn-risk hypotheses, and the business memo.

In [ ]:
from build_part1 import find_data_dir, load_data, build_feature_table

data_dir = find_data_dir()
data = load_data(data_dir)
feature_table = build_feature_table(data)

{name: df.shape for name, df in data.items()}


## Package Inspection

| file_name                           | file_type   | rows   | columns   | grain                     | note                                                                                                          |
|:------------------------------------|:------------|:-------|:----------|:--------------------------|:--------------------------------------------------------------------------------------------------------------|
| DATA_DICTIONARY.md                  | md          | —      | —         | Reference / documentation | Included in the package and reviewed for project context                                                      |
| STUDENT_FACING_PROBLEM_STATEMENT.md | md          | —      | —         | Reference / documentation | Included in the package and reviewed for project context                                                      |
| churn_labels.csv                    | csv         | 2400   | 4         | 1 row per customer        | Target table with churn label and split assignment                                                            |
| customers.csv                       | csv         | 2400   | 9         | 1 row per customer        | Raw customer profile table                                                                                    |
| intervention_history.csv            | csv         | 2400   | 5         | 1 row per customer        | Most recent pre-snapshot campaign/intervention per customer                                                   |
| orders.csv                          | csv         | 10009  | 10        | 1 row per order line      | Raw transaction table; includes post-snapshot rows used only for label construction                           |
| rfm_modeling_snapshot.csv           | csv         | 2400   | 29        | 1 row per customer        | Derived modeling table shipped in the package; inspected here but not used to derive Part 1 business findings |
| support_tickets.csv                 | csv         | 1921   | 8         | 1 row per ticket          | Raw support interaction table                                                                                 |
| web_events_snapshot.csv             | csv         | 2400   | 10        | 1 row per customer        | Raw 30-day app/web engagement snapshot                                                                        |

## Schema Summary

| dataset                   |   rows |   columns | primary_key   | date_columns   | sample_columns                                                                                    |
|:--------------------------|-------:|----------:|:--------------|:---------------|:--------------------------------------------------------------------------------------------------|
| customers.csv             |   2400 |         9 | customer_id   | signup_date    | customer_id, signup_date, city_tier, age_group, acquisition_channel, loyalty_tier ...             |
| orders.csv                |  10009 |        10 | order_id      | order_date     | order_id, customer_id, order_date, category, quantity, gross_amount ...                           |
| support_tickets.csv       |   1921 |         8 | ticket_id     | ticket_date    | ticket_id, customer_id, ticket_date, issue_type, support_channel, resolution_hours ...            |
| web_events_snapshot.csv   |   2400 |        10 | customer_id   | snapshot_date  | customer_id, snapshot_date, sessions_30d, product_views_30d, cart_adds_30d, wishlist_adds_30d ... |
| churn_labels.csv          |   2400 |         4 | customer_id   | snapshot_date  | customer_id, snapshot_date, churn_next_60d, split                                                 |
| rfm_modeling_snapshot.csv |   2400 |        29 | customer_id   | snapshot_date  | customer_id, snapshot_date, city_tier, age_group, acquisition_channel, loyalty_tier ...           |
| intervention_history.csv  |   2400 |         5 | customer_id   | snapshot_date  | customer_id, snapshot_date, last_campaign_received, last_campaign_cost, manual_priority_bucket    |

## Join Flow

The analysis is done at customer level. Raw transactional and interaction tables are aggregated first and then left-joined back to the 2,400-customer base.

| step                            | join_key    |   source_rows |   distinct_customers |   matched_customers |   coverage_pct | note                                                                           |
|:--------------------------------|:------------|--------------:|---------------------:|--------------------:|---------------:|:-------------------------------------------------------------------------------|
| Base customer universe          | customer_id |          2400 |                 2400 |                2400 |          100   | Starting point for all customer-level analysis                                 |
| Latest pre-snapshot orders      | customer_id |          8137 |                 2400 |                2400 |          100   | Used for total order depth and recency                                         |
| Orders in last 180 days         | customer_id |          3830 |                 2098 |                2098 |           87.4 | Used for frequency, monetary value, discounts, returns, and category diversity |
| 30-day web/app snapshot         | customer_id |          2400 |                 2400 |                2400 |          100   | One row per customer; clean left join                                          |
| Support tickets in last 90 days | customer_id |           550 |                  498 |                 498 |           20.8 | Sparse join; non-ticket customers are kept and later zero-filled               |
| Intervention history snapshot   | customer_id |          2400 |                 2400 |                2400 |          100   | Campaign and manual-priority context                                           |
| Observed churn labels           | customer_id |          2400 |                 2400 |                2400 |          100   | Joined only for analysis outputs, not as a model feature                       |
| Final joined feature table      | customer_id |          2400 |                 2400 |                2400 |          100   | Customer-level analysis table used for EDA and churn-pattern slicing           |

In [ ]:
from build_part1 import SNAPSHOT_DATE

customers = data["customers"].copy()
orders = data["orders"].copy()
support = data["support_tickets"].copy()
web = data["web_events_snapshot"].copy()
labels = data["churn_labels"].copy()
interventions = data["intervention_history"].copy()

pre_orders = orders.loc[orders["order_date"] <= SNAPSHOT_DATE].copy()
orders_180d = pre_orders.loc[pre_orders["order_date"] >= SNAPSHOT_DATE - pd.Timedelta(days=180)].copy()
tickets_90d = support.loc[support["ticket_date"] >= SNAPSHOT_DATE - pd.Timedelta(days=90)].copy()

order_agg = pre_orders.groupby("customer_id").agg(last_order_date=("order_date", "max")).reset_index()
order_180 = orders_180d.groupby("customer_id").agg(
    frequency_180d=("order_id", "nunique"),
    monetary_180d=("gross_amount", "sum"),
).reset_index()
support_90 = tickets_90d.groupby("customer_id").agg(ticket_count_90d=("ticket_id", "count")).reset_index()

joined = (
    customers.merge(order_agg, on="customer_id", how="left")
    .merge(order_180, on="customer_id", how="left")
    .merge(web, on="customer_id", how="left")
    .merge(support_90, on="customer_id", how="left")
    .merge(interventions, on="customer_id", how="left")
    .merge(labels[["customer_id", "churn_next_60d"]], on="customer_id", how="left")
)

joined.shape, joined["customer_id"].nunique()


## Joined Customer Feature Sample

This sample shows the customer-level analysis table after joins and pre-EDA feature preparation.

| customer_id   |   recency_days |   frequency_180d |   monetary_180d |   sessions_30d |   ticket_count_90d | last_campaign_received   |   churn_next_60d |
|:--------------|---------------:|-----------------:|----------------:|---------------:|-------------------:|:-------------------------|-----------------:|
| CUST00001     |            107 |                1 |          362.73 |              1 |                  0 | welcome_offer            |                1 |
| CUST00002     |             40 |                1 |          581    |              8 |                  1 | free_shipping            |                0 |
| CUST00003     |            171 |                1 |          649.98 |              1 |                  0 | none                     |                1 |
| CUST00004     |            131 |                1 |         1604.04 |              1 |                  0 | free_shipping            |                1 |
| CUST00005     |             38 |                3 |         1781.9  |             18 |                  0 | welcome_offer            |                0 |
| CUST00006     |             51 |                4 |         2989.58 |              2 |                  2 | none                     |                0 |
| CUST00007     |              3 |                1 |          719.33 |             11 |                  0 | free_shipping            |                0 |
| CUST00008     |             47 |                3 |         2449.11 |              2 |                  0 | bundle_discount          |                0 |
| CUST00009     |             31 |                1 |          376.85 |             11 |                  0 | none                     |                0 |
| CUST00010     |              9 |                1 |          636.8  |             13 |                  0 | none                     |                0 |
| CUST00011     |              1 |                1 |          508.08 |             10 |                  0 | new_launch               |                0 |
| CUST00012     |             50 |                1 |          978.88 |              3 |                  0 | none                     |                1 |

# Data Quality Report

## Package Manifest

| file_name                           | file_type   | rows   | columns   | grain                     | note                                                                                                          |
|:------------------------------------|:------------|:-------|:----------|:--------------------------|:--------------------------------------------------------------------------------------------------------------|
| DATA_DICTIONARY.md                  | md          | —      | —         | Reference / documentation | Included in the package and reviewed for project context                                                      |
| STUDENT_FACING_PROBLEM_STATEMENT.md | md          | —      | —         | Reference / documentation | Included in the package and reviewed for project context                                                      |
| churn_labels.csv                    | csv         | 2400   | 4         | 1 row per customer        | Target table with churn label and split assignment                                                            |
| customers.csv                       | csv         | 2400   | 9         | 1 row per customer        | Raw customer profile table                                                                                    |
| intervention_history.csv            | csv         | 2400   | 5         | 1 row per customer        | Most recent pre-snapshot campaign/intervention per customer                                                   |
| orders.csv                          | csv         | 10009  | 10        | 1 row per order line      | Raw transaction table; includes post-snapshot rows used only for label construction                           |
| rfm_modeling_snapshot.csv           | csv         | 2400   | 29        | 1 row per customer        | Derived modeling table shipped in the package; inspected here but not used to derive Part 1 business findings |
| support_tickets.csv                 | csv         | 1921   | 8         | 1 row per ticket          | Raw support interaction table                                                                                 |
| web_events_snapshot.csv             | csv         | 2400   | 10        | 1 row per customer        | Raw 30-day app/web engagement snapshot                                                                        |

## Loaded Dataset Inspection

| dataset                   |   rows |   columns | primary_key   | date_columns   | sample_columns                                                                                    |
|:--------------------------|-------:|----------:|:--------------|:---------------|:--------------------------------------------------------------------------------------------------|
| customers.csv             |   2400 |         9 | customer_id   | signup_date    | customer_id, signup_date, city_tier, age_group, acquisition_channel, loyalty_tier ...             |
| orders.csv                |  10009 |        10 | order_id      | order_date     | order_id, customer_id, order_date, category, quantity, gross_amount ...                           |
| support_tickets.csv       |   1921 |         8 | ticket_id     | ticket_date    | ticket_id, customer_id, ticket_date, issue_type, support_channel, resolution_hours ...            |
| web_events_snapshot.csv   |   2400 |        10 | customer_id   | snapshot_date  | customer_id, snapshot_date, sessions_30d, product_views_30d, cart_adds_30d, wishlist_adds_30d ... |
| churn_labels.csv          |   2400 |         4 | customer_id   | snapshot_date  | customer_id, snapshot_date, churn_next_60d, split                                                 |
| rfm_modeling_snapshot.csv |   2400 |        29 | customer_id   | snapshot_date  | customer_id, snapshot_date, city_tier, age_group, acquisition_channel, loyalty_tier ...           |
| intervention_history.csv  |   2400 |         5 | customer_id   | snapshot_date  | customer_id, snapshot_date, last_campaign_received, last_campaign_cost, manual_priority_bucket    |

## Missing Values

| dataset                   | column       |   missing_rows |   missing_pct |
|:--------------------------|:-------------|---------------:|--------------:|
| customers.csv             | loyalty_tier |           1386 |          57.8 |
| rfm_modeling_snapshot.csv | loyalty_tier |           1386 |          57.8 |
| customers.csv             | skin_type    |            401 |          16.7 |
| orders.csv                | rating       |             80 |           0.8 |

## Duplicate and Duplicate-Like Records

| dataset                   |   exact_duplicate_rows |   duplicate_primary_keys |   duplicate_like_rows |
|:--------------------------|-----------------------:|-------------------------:|----------------------:|
| customers.csv             |                      0 |                        0 |                     0 |
| orders.csv                |                      0 |                        0 |                    12 |
| support_tickets.csv       |                      0 |                        0 |                     0 |
| web_events_snapshot.csv   |                      0 |                        0 |                     0 |
| churn_labels.csv          |                      0 |                        0 |                     0 |
| rfm_modeling_snapshot.csv |                      0 |                        0 |                     0 |
| intervention_history.csv  |                      0 |                        0 |                     0 |

`orders.csv` contains intentionally duplicate-like rows whose `order_id` ends with `_DUP`. Those should be removed or collapsed into their base order before any customer aggregation.

Sample duplicate-like rows:

| order_id      | base_order_id   | customer_id   | order_date          |   gross_amount |
|:--------------|:----------------|:--------------|:--------------------|---------------:|
| ORD008249_DUP | ORD008249       | CUST00153     | 2025-11-04 00:00:00 |         321.31 |
| ORD002124_DUP | ORD002124       | CUST00628     | 2025-03-18 00:00:00 |         410.04 |
| ORD002862_DUP | ORD002862       | CUST00837     | 2025-07-12 00:00:00 |         952.02 |
| ORD002916_DUP | ORD002916       | CUST00848     | 2025-09-26 00:00:00 |         547.18 |
| ORD002970_DUP | ORD002970       | CUST00869     | 2024-12-22 00:00:00 |         818.64 |
| ORD008836_DUP | ORD008836       | CUST00875     | 2025-10-23 00:00:00 |         711.2  |
| ORD003897_DUP | ORD003897       | CUST01140     | 2025-04-14 00:00:00 |         769.96 |
| ORD004577_DUP | ORD004577       | CUST01335     | 2025-02-12 00:00:00 |         533.07 |

## Join / Key Issues

| dataset                   | primary_key   |   duplicate_primary_keys | orphan_customer_ids   |   distinct_customer_ids |
|:--------------------------|:--------------|-------------------------:|:----------------------|------------------------:|
| customers.csv             | customer_id   |                        0 | —                     |                    2400 |
| orders.csv                | order_id      |                        0 | 0                     |                    2400 |
| support_tickets.csv       | ticket_id     |                        0 | 0                     |                    1247 |
| web_events_snapshot.csv   | customer_id   |                        0 | 0                     |                    2400 |
| churn_labels.csv          | customer_id   |                        0 | 0                     |                    2400 |
| rfm_modeling_snapshot.csv | customer_id   |                        0 | 0                     |                    2400 |
| intervention_history.csv  | customer_id   |                        0 | 0                     |                    2400 |

All customer-linked tables join back cleanly to the 2,400-customer universe; the main integrity risk is duplicate handling rather than orphaned IDs.

## Invalid or Unusual Values

| check                                                     |   count | recommendation                                                                                  |
|:----------------------------------------------------------|--------:|:------------------------------------------------------------------------------------------------|
| orders.rating outside 1-5                                 |       0 | Treat as invalid rating values if any appear.                                                   |
| orders.discount_pct outside 0.0-0.7                       |       0 | Clamp or investigate pricing logic if the count is non-zero.                                    |
| orders.delivery_days outside 1-11                         |       0 | Review fulfillment timestamp logic if values fall outside the documented range.                 |
| orders.gross_amount < 0                                   |       0 | Negative order values should be treated as invalid unless explicitly documented as adjustments. |
| support_tickets.sentiment_score outside -1 to 1           |       0 | Recompute or clip sentiment scores if values fall outside the scoring range.                    |
| support_tickets.resolution_hours <= 0                     |       0 | Resolution time should be positive for closed tickets.                                          |
| Negative counts in web/app activity snapshot              |       0 | Activity metrics should be non-negative; audit source event processing if not.                  |
| last_campaign_received = none but last_campaign_cost != 0 |     404 | Reset spend to 0 or audit the CRM export before ROI analysis.                                   |
| last_campaign_received != none but last_campaign_cost = 0 |     377 | Treat campaign spend as incomplete or backfill missing costs.                                   |

## Date Consistency Checks

| check                                                   |   count | recommendation                                                                     |
|:--------------------------------------------------------|--------:|:-----------------------------------------------------------------------------------|
| customers.signup_date after snapshot date               |       0 | Future-dated signups should be audited before lifecycle analysis.                  |
| orders.order_date before customer signup_date           |       0 | Transactions before signup usually indicate join or source-system timing problems. |
| support_tickets.ticket_date before customer signup_date |       0 | Support activity should not predate account creation.                              |
| support_tickets.ticket_date after snapshot date         |       0 | Ticket history should be snapshot-aligned for Part 1 and model-safe feature work.  |
| web_events_snapshot.snapshot_date != 2025-09-30         |       0 | Snapshot tables should share the same reference date.                              |
| churn_labels.snapshot_date != 2025-09-30                |       0 | Labels must align to the shared snapshot boundary.                                 |
| intervention_history.snapshot_date != 2025-09-30        |       0 | Intervention history should align to the same snapshot date.                       |
| rfm_modeling_snapshot.snapshot_date != 2025-09-30       |       0 | The derived modeling table should align to the raw-snapshot reference date.        |

## Leakage-Sensitive Columns and Rows

| dataset                   | column_or_rows                     | why_risky                                                                                         |
|:--------------------------|:-----------------------------------|:--------------------------------------------------------------------------------------------------|
| orders.csv                | Rows where order_date > 2025-09-30 | These rows occur after the modeling snapshot and can leak future purchase behavior into features. |
| churn_labels.csv          | churn_next_60d                     | This is the target label and must never be used as an input feature.                              |
| churn_labels.csv          | split                              | This is evaluation metadata, not customer behavior.                                               |
| rfm_modeling_snapshot.csv | churn_next_60d                     | Target copy embedded in the modeling snapshot.                                                    |
| rfm_modeling_snapshot.csv | split                              | Pre-assigned fold metadata; safe for evaluation only.                                             |
| rfm_modeling_snapshot.csv | snapshot_date                      | This column is constant here but should not be treated as a predictive behavior variable.         |

## Outlier Audit

| dataset             | column           |   upper_iqr_fence |   outlier_rows |     p99 |   max_value |
|:--------------------|:-----------------|------------------:|---------------:|--------:|------------:|
| orders.csv          | gross_amount     |            1619.3 |            536 | 2308.62 |     24789.4 |
| support_tickets.csv | resolution_hours |              64.9 |              9 |   59.96 |        74.6 |

Top gross-amount outliers:

| order_id   | customer_id   | order_date          | category   |   gross_amount |   discount_pct |
|:-----------|:--------------|:--------------------|:-----------|---------------:|---------------:|
| ORD006374  | CUST01868     | 2025-03-29 00:00:00 | Skin Care  |       24789.4  |           0.13 |
| ORD000701  | CUST00211     | 2024-11-27 00:00:00 | Fragrance  |       22719.5  |           0.25 |
| ORD007206  | CUST02106     | 2024-07-13 00:00:00 | Fragrance  |       15957.5  |           0.37 |
| ORD009649  | CUST01988     | 2025-10-25 00:00:00 | Fragrance  |       12312.1  |           0.04 |
| ORD004428  | CUST01295     | 2025-05-01 00:00:00 | Baby Care  |       10643.8  |           0.04 |
| ORD004650  | CUST01360     | 2024-10-09 00:00:00 | Fragrance  |        8777.2  |           0.47 |
| ORD005399  | CUST01584     | 2024-12-31 00:00:00 | Fragrance  |        8022.5  |           0.17 |
| ORD007765  | CUST02287     | 2025-06-22 00:00:00 | Fragrance  |        3746.76 |           0.08 |
| ORD000500  | CUST00159     | 2024-06-13 00:00:00 | Fragrance  |        3376.32 |           0.17 |
| ORD001120  | CUST00324     | 2024-12-30 00:00:00 | Fragrance  |        3341.27 |           0.16 |

## Treatment Recommendations

1. Deduplicate the 12 `_DUP` rows in `orders.csv` before any order-count or spend aggregation.
2. Enforce the snapshot boundary strictly: post-snapshot orders belong to label construction, not model features.
3. Keep true missingness visible for `loyalty_tier`, `skin_type`, and `rating`; encode it rather than silently dropping rows.
4. Winsorize or log-transform `gross_amount` before modelling because spend outliers are extreme enough to dominate averages.
5. Audit `intervention_history.csv` before campaign ROI work because campaign-cost fields are internally inconsistent for hundreds of customers.

## Business-Facing Readout

1. Recency is the strongest warning sign: customers with 121+ day recency churn at **89.2%** versus **11.7%** for customers who purchased in the last month.
2. Thin recent order depth matters: customers with only one order in the last 180 days churn at **61.6%**, while the 5+ order group drops to **14.8%**.
3. Low spend depth is risky: the bottom spend quartile churns at **75.7%** versus **20.7%** for the top quartile.
4. Return-heavy customers are fragile: customers with 50%+ return rates churn at **75.0%**.
5. Acquisition quality varies: Google Search customers churn at **50.4%**, materially above the **39.8%** seen in Organic acquisition.


## Visual EDA

### Observed 60-Day Churn Distribution
![Observed 60-Day Churn Distribution](charts/01_churn_distribution.png)

Churn affects 47.0% of the 2,400-customer base, so the company has a real retention problem rather than a fringe outlier cohort.

### Monthly Order Volume and the Leakage Boundary
![Monthly Order Volume and the Leakage Boundary](charts/02_monthly_orders_and_snapshot.png)

The package contains 1,872 post-snapshot order rows. They help explain label construction but cannot be used as model features.

### Churn Rate by Acquisition Channel
![Churn Rate by Acquisition Channel](charts/03_acquisition_channel_vs_churn.png)

Google Search customers churn at 50.4% versus 39.8% for Organic customers.

### Churn Rate by Order Frequency in the Last 180 Days
![Churn Rate by Order Frequency in the Last 180 Days](charts/04_frequency_vs_churn.png)

Customers with only one recent order churn at 61.6%, while customers with 5+ recent orders fall to 14.8%.

### Churn Rate by 180-Day Spend Quartile
![Churn Rate by 180-Day Spend Quartile](charts/05_monetary_vs_churn.png)

The lowest spend quartile churns at 75.7%, compared with 20.7% for the top quartile.

### Customer Churn Rate by Support Issue Type
![Customer Churn Rate by Support Issue Type](charts/06_support_issue_vs_churn.png)

`wrong_item` is the highest-churn support issue cohort at 45.5%. This chart is customer-level, so one customer can appear in more than one issue group.

### Churn Rate by Return Rate in the Last 180 Days
![Churn Rate by Return Rate in the Last 180 Days](charts/07_return_rate_vs_churn.png)

Customers with return rates above 50% churn at 75.0%, versus 47.2% for customers with no recent returns.

### Churn Rate by 30-Day Session Activity
![Churn Rate by 30-Day Session Activity](charts/08_sessions_vs_churn.png)

Customers with only 0-2 sessions churn at 64.5%, while customers with 9+ sessions drop to 24.7%.

### Churn Rate by Most Recent Campaign Received
![Churn Rate by Most Recent Campaign Received](charts/09_campaign_history_vs_churn.png)

`new_launch` recipients churn at 51.0%, versus 45.2% for customers with no recent campaign.

### Churn Rate by Days Since Last Order
![Churn Rate by Days Since Last Order](charts/10_recency_vs_churn.png)

Customers with recency over 120 days churn at 89.2%, versus 11.7% for customers who purchased in the last month.

### Churn Rate by Category Diversity in the Last 180 Days
![Churn Rate by Category Diversity in the Last 180 Days](charts/11_category_diversity_vs_churn.png)

One-category shoppers churn at 58.6%, while customers buying across three categories drop to 16.9%.

### Churn Rate by Reopened Ticket Share (Ticketed Customers Only)
![Churn Rate by Reopened Ticket Share (Ticketed Customers Only)](charts/12_reopened_tickets_vs_churn.png)

Among customers who raised support tickets, those with 50%+ reopened tickets churn at 41.9%.

## Exploratory Analysis Tables

### Customer Demographics / Profile

**Churn by acquisition channel**

| acquisition_channel   |   customers |   churn_rate |   churn_rate_pct |
|:----------------------|------------:|-------------:|-----------------:|
| Google Search         |         466 |     0.504292 |             50.4 |
| Instagram             |         517 |     0.499033 |             49.9 |
| Marketplace           |         456 |     0.491228 |             49.1 |
| Influencer            |         231 |     0.47619  |             47.6 |
| Referral              |         396 |     0.421717 |             42.2 |
| Organic               |         334 |     0.398204 |             39.8 |

**Churn by age group**

| age_group   |   customers |   churn_rate |   churn_rate_pct |
|:------------|------------:|-------------:|-----------------:|
| 35-44       |         534 |     0.483146 |             48.3 |
| 25-34       |        1045 |     0.47177  |             47.2 |
| 45+         |         261 |     0.463602 |             46.4 |
| 18-24       |         560 |     0.455357 |             45.5 |

**Churn by city tier**

| city_tier   |   customers |   churn_rate |   churn_rate_pct |
|:------------|------------:|-------------:|-----------------:|
| Tier 2      |         870 |     0.477011 |             47.7 |
| Tier 1      |        1005 |     0.473632 |             47.4 |
| Tier 3      |         525 |     0.449524 |             45   |

**Churn by marketing consent**

| marketing_consent   |   customers |   churn_rate |   churn_rate_pct |
|:--------------------|------------:|-------------:|-----------------:|
| No                  |         640 |     0.479687 |             48   |
| Yes                 |        1760 |     0.465909 |             46.6 |

### Order Behaviour

**Churn by recency**

| recency_bucket   |   customers |   churn_rate |   churn_rate_pct |
|:-----------------|------------:|-------------:|-----------------:|
| 0-30             |         699 |     0.11731  |             11.7 |
| 31-60            |         441 |     0.303855 |             30.4 |
| 61-120           |         595 |     0.534454 |             53.4 |
| 121+             |         665 |     0.891729 |             89.2 |

**Churn by recent order frequency**

| frequency_bucket   |   customers |   churn_rate |   churn_rate_pct |
|:-------------------|------------:|-------------:|-----------------:|
| 1                  |        1376 |     0.615552 |             61.6 |
| 2                  |         579 |     0.35924  |             35.9 |
| 3-4                |         384 |     0.164062 |             16.4 |
| 5+                 |          61 |     0.147541 |             14.8 |

### Monetary Behaviour

**Churn by spend quartile**

| monetary_bucket   |   customers |   churn_rate |   churn_rate_pct |
|:------------------|------------:|-------------:|-----------------:|
| Q1 Low            |         600 |     0.756667 |             75.7 |
| Q2                |         600 |     0.508333 |             50.8 |
| Q3                |         600 |     0.406667 |             40.7 |
| Q4 High           |         600 |     0.206667 |             20.7 |

**Spend summary by observed outcome**

| churn_next_60d   |   customers |   avg_monetary_180d |   median_monetary_180d |   avg_frequency_180d |
|:-----------------|------------:|--------------------:|-----------------------:|---------------------:|
| Retained         |        1273 |             1556.51 |                1306.09 |                 2.04 |
| Churned          |        1127 |              764.81 |                 568.21 |                 1.1  |

### Support-Ticket Issues

**Customer-level churn by support issue type**

| issue_type       |   customers_with_issue |   ticket_count |   customer_churn_pct |   avg_resolution_hours |   avg_sentiment |   reopened_pct |
|:-----------------|-----------------------:|---------------:|---------------------:|-----------------------:|----------------:|---------------:|
| wrong_item       |                    200 |            213 |                 45.5 |                  20.9  |           -0.41 |           12.7 |
| product_reaction |                    188 |            194 |                 45.2 |                  36.54 |           -0.6  |           26.8 |
| general_query    |                    298 |            324 |                 44   |                  20.83 |           -0.42 |           16.4 |
| damaged_item     |                    261 |            277 |                 43.3 |                  19.18 |           -0.31 |           15.2 |
| payment_issue    |                    181 |            191 |                 43.1 |                  20.14 |           -0.35 |           15.7 |
| refund_delay     |                    318 |            345 |                 42.8 |                  36.46 |           -0.65 |           22.3 |
| late_delivery    |                    335 |            377 |                 42.7 |                  20.13 |           -0.35 |           15.6 |

### Return / Refund Behaviour

| return_rate_bucket   |   customers |   churn_rate |   churn_rate_pct |
|:---------------------|------------:|-------------:|-----------------:|
| 0                    |        2142 |    0.471522  |             47.2 |
| 0-25%                |          46 |    0.0869565 |              8.7 |
| 25-50%               |         128 |    0.390625  |             39.1 |
| 50%+                 |          84 |    0.75      |             75   |

### Web / App Activity

**Churn by 30-day session activity**

| sessions_bucket   |   customers |   churn_rate |   churn_rate_pct |
|:------------------|------------:|-------------:|-----------------:|
| 0-2               |         782 |     0.644501 |             64.5 |
| 3-5               |         599 |     0.509182 |             50.9 |
| 6-8               |         460 |     0.391304 |             39.1 |
| 9+                |         559 |     0.246869 |             24.7 |

**Churn by category diversity**

| category_diversity_bucket   |   customers |   churn_rate |   churn_rate_pct |
|:----------------------------|------------:|-------------:|-----------------:|
| 1                           |        1522 |     0.586071 |             58.6 |
| 2                           |         607 |     0.314662 |             31.5 |
| 3                           |         219 |     0.16895  |             16.9 |
| 4+                          |          52 |     0.134615 |             13.5 |

### Campaign / Intervention History

**Churn by most recent campaign**

| last_campaign_received   |   customers |   avg_campaign_cost |   churn_rate_pct |
|:-------------------------|------------:|--------------------:|-----------------:|
| new_launch               |         498 |                18.3 |             51   |
| bundle_discount          |         473 |                18.7 |             46.9 |
| free_shipping            |         469 |                19.5 |             46.3 |
| welcome_offer            |         453 |                17.3 |             45.3 |
| none                     |         507 |                18.5 |             45.2 |

**Churn by manual CRM priority bucket**

| manual_priority_bucket   |   customers |   avg_campaign_cost |   churn_rate_pct |
|:-------------------------|------------:|--------------------:|-----------------:|
| high                     |        1163 |                18.5 |             74.7 |
| medium                   |         749 |                18.4 |             27.9 |
| low                      |         488 |                18.5 |             10   |


## Churn-Risk Hypotheses

1. **Stale recency is the strongest churn signal.** Supported by *Churn Rate by Days Since Last Order*. Customers with 121+ day recency churn at **89.2%**, versus **11.7%** for customers who ordered in the last month.
2. **Weak recent web/app activity is an early churn warning.** Supported by *Churn Rate by 30-Day Session Activity*. Customers with only 0-2 sessions churn at **64.5%**, while the 9+ session group drops to **24.7%**.
3. **Thin recent order frequency signals fragile habit formation.** Supported by *Churn Rate by Order Frequency in the Last 180 Days*. Customers with a single recent order churn at **61.6%**, compared with **14.8%** for the 5+ order cohort.
4. **Low spend depth is associated with higher churn.** Supported by *Churn Rate by 180-Day Spend Quartile*. The lowest spend quartile churns at **75.7%**, versus **20.7%** for the top quartile.
5. **Acquisition quality differs by channel.** Supported by *Churn Rate by Acquisition Channel*. Google Search customers churn at **50.4%** and Instagram customers at **49.9%**, both above Organic at **39.8%**.
6. **Returns and unresolved service friction likely suppress repeat purchase intent.** Supported by *Churn Rate by Return Rate in the Last 180 Days* and *Churn Rate by Reopened Ticket Share*. Customers with 50%+ return rates churn at **75.0%**, and ticketed customers with 50%+ reopened cases churn at **41.9%**.


# Business Memo

## To
Product, CRM, and Customer Support Leaders

## Subject
What the company should investigate before launching a retention campaign

## Executive Summary

Churn is material at **47.0%** of the customer base. The main risk pattern is inactivity: customers with 121+ day recency churn at **89.2%**, and customers with only 0-2 sessions in the last month churn at **64.5%**.

## What To Investigate Before Spending Retention Budget

1. **Split dormant customers by recoverability, not just by age since last order.** Customers with 121+ day recency are extremely high risk, but those who still browse occasionally are more recoverable than those with no current activity.
2. **Treat paid-acquisition cohorts differently from Organic or Referral cohorts.** Google Search customers churn at **50.4%** and Instagram customers also sit near 50%, materially above Organic at **39.8%**.
3. **Protect recent value before chasing the coldest names.** One-order customers churn at **61.6%**, and the lowest spend quartile churns at **75.7%**. The team should separate low-value churn from high-value churn instead of treating them with the same offer.
4. **Diagnose operational friction before defaulting to discounts.** Customers with 50%+ return rates churn at **75.0%**. Among customers who raised tickets, the 50%+ reopened group churns at **41.9%**. `wrong_item` is the highest-churn issue cohort at **45.5%**.
5. **Review whether the current campaign mix is matched to risk.** Customers whose latest touch was `new_launch` still churn at **51.0%**, only modestly different from the **45.2%** seen for customers with no campaign at all. That pattern suggests targeting and offer design need review before scaling spend.

## Recommended Pilot Structure

Run two controlled pilots before a full launch:

1. A **service-recovery lane** for customers with returns, reopened tickets, or severe support issues.
2. A **reactivation lane** for high-value customers showing stale recency and weak digital activity.

Both pilots should include holdouts so the team measures incremental lift rather than raw return rate.


In [ ]:
# Rebuild the part outputs from the command line:
# python build_part1.py
